# LJ Dev Commerce - Phase 4

# II. Supplier ETL

## 1. Supplier Source Data

The Supplier dataset represents a simulated extract from the Procurement / ERP source system.

This notebook documents the **raw-data profiling stage** before any cleaning, transformation, validation, or PostgreSQL loading.

### Profiling objectives

- Understand the source structure and data types
- Identify missing, blank, and whitespace values
- Detect duplicate records and duplicate business identifiers
- Detect semantic field types such as dates, emails, phones, identifiers, and categorical fields
- Identify numeric and date-quality issues
- Produce a reusable profiling result that will drive the transformation stage

**Important:** No source data is modified during this profiling stage.

## 2. Extract Raw Supplier Data

In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import sys

# Read the raw Supplier CSV
procurement_supplier_master_raw = pd.read_csv(
    "../data/02_Procurement_ERP/procurement_supplier_master.csv"
)

procurement_supplier_master_raw.head()

,SupplierCode,SupplierName,ContactPerson,EmailAddress,PhoneNumber,StreetAddress,City,Country,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,SUP001,ABC Trading LLC,John Smith,supplier1@email.com,+971 50 111 2201,Al Quoz,Dubai,UAE,True,2026-07-01,procurement_admin,2026-07-15,procurement_admin
1,SUP002,TechWorld Middle East,Maria Khan,sales@techworld.ae,+971501112202,Jebel Ali,Dubai,UAE,True,2026-07-01,procurement_admin,2026-07-14,procurement_admin
2,SUP003,Gulf Electronics FZCO,Ahmed Rahman,CONTACT@GULFELECTRONICS.AE,+971 50 111 2203,Dubai Silicon Oasis,Dubai,UAE,True,2026-07-02,procurement_admin,2026-07-15,procurement_admin
3,SUP004,Asia Components Trading,Li Wei,asia.components@email.com,+971 50 111 2204,Deira,Dubai,UAE,True,2026-07-02,procurement_admin,2026-07-13,procurement_admin
4,SUP005,Global Devices Ltd,Sarah Lee,global.devices@email.com,+971 50 111 2205,Business Bay,Dubai,UAE,True,2026-07-03,procurement_admin,2026-07-15,procurement_admin


In [4]:
# Confirm the raw dataset size
print("Rows:", procurement_supplier_master_raw.shape[0])
print("Columns:", procurement_supplier_master_raw.shape[1])

Rows: 9
Columns: 13


In [5]:
# Review the source columns and Pandas data types
display(
    pd.DataFrame({
        "Column": procurement_supplier_master_raw.columns,
        "DataType": procurement_supplier_master_raw.dtypes.astype(str).values
    })
)

,Column,DataType
0,SupplierCode,object
1,SupplierName,object
2,ContactPerson,object
3,EmailAddress,object
4,PhoneNumber,object
5,StreetAddress,object
6,City,object
7,Country,object
8,ActiveFlag,bool
9,CreatedOn,object


## 3. Initial Raw-Data Quality Checks

Before running the reusable profiler, a small set of direct checks establishes the baseline condition of the source data.

In [6]:
# Missing values by column
missing_profile = procurement_supplier_master_raw.isna().sum()
missing_profile

SupplierCode      0
SupplierName      0
ContactPerson     0
EmailAddress      0
PhoneNumber       0
StreetAddress     0
City              0
Country           0
ActiveFlag        0
CreatedOn         0
CreatedByUser     0
ModifiedOn        0
ModifiedByUser    0
dtype: int64

In [7]:
# Exact duplicate rows
exact_duplicate_count = procurement_supplier_master_raw.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_count)

# Show duplicate records for investigation
procurement_supplier_master_raw[procurement_supplier_master_raw.duplicated(keep=False)]

Exact duplicate rows: 1


,SupplierCode,SupplierName,ContactPerson,EmailAddress,PhoneNumber,StreetAddress,City,Country,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
6,SUP007,Nova Tech Distribution,Daniel Cruz,nova.tech@email.com,+971 50 111 2207,Sharjah Industrial Area,sharjah,UAE,True,2026-07-04,procurement_admin,2026-07-15,procurement_admin
8,SUP007,Nova Tech Distribution,Daniel Cruz,nova.tech@email.com,+971 50 111 2207,Sharjah Industrial Area,sharjah,UAE,True,2026-07-04,procurement_admin,2026-07-15,procurement_admin


In [8]:
# Blank and whitespace-only values in text columns
text_columns = procurement_supplier_master_raw.select_dtypes(include="object").columns

blank_whitespace_profile = {
    column: int(
        procurement_supplier_master_raw[column].astype(str).str.strip().eq("").sum()
    )
    for column in text_columns
}

blank_whitespace_profile

{'SupplierCode': 0,
 'SupplierName': 0,
 'ContactPerson': 0,
 'EmailAddress': 0,
 'PhoneNumber': 0,
 'StreetAddress': 0,
 'City': 0,
 'Country': 0,
 'CreatedOn': 0,
 'CreatedByUser': 0,
 'ModifiedOn': 0,
 'ModifiedByUser': 0}

## 4. Data Profiling

The reusable profiler in `profiler/data_profiler_v1.py` performs the detailed semantic, statistical, and data-quality profiling.

The notebook uses the profiler as a separate module so that the profiling logic is reusable across datasets rather than being duplicated inside the notebook.

In [9]:
# Import and reload the current reusable profiler
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
from profiler import data_profiler_v1 as profiler

importlib.reload(profiler)

print("Profiler:", profiler.__file__)
print("Configuration:", profiler.DEFAULT_CONFIG)

Profiler: c:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\profiler\data_profiler_v1.py
Configuration: {'date_detection_threshold': 0.8, 'numeric_detection_threshold': 0.8, 'email_detection_threshold': 0.8, 'phone_detection_threshold': 0.8, 'categorical_unique_ratio': 0.2, 'outlier_iqr_multiplier': 1.5, 'required_columns': [], 'unique_columns': [], 'non_negative_columns': []}


In [10]:
# Run the complete profiler against the untouched raw Supplier dataset
profile_results = profiler.profile_dataset(procurement_supplier_master_raw)

print("Profiler sections:")
print(list(profile_results.keys()))

Profiler sections:
['field_types', 'detection_details', 'general', 'text', 'categorical', 'numeric', 'date', 'patterns', 'issues', 'configuration']


### 4.1 Semantic Field-Type Detection

In [12]:
profile_results["field_types"]

SupplierCode       identifier
SupplierName             text
ContactPerson            text
EmailAddress            email
PhoneNumber             phone
StreetAddress            text
City                     text
Country           categorical
ActiveFlag            boolean
CreatedOn                date
CreatedByUser     categorical
ModifiedOn               date
ModifiedByUser    categorical
Name: DetectedFieldType, dtype: object

### 4.2 General Dataset Profile

In [13]:
profile_results["general"]

{'rows': 9,
 'columns': 13,
 'exact_duplicate_rows': 1,
 'column_profile':                DataType DetectedFieldType  MissingCount  MissingPercent  \
 SupplierCode     object        identifier             0             0.0   
 SupplierName     object              text             0             0.0   
 ContactPerson    object              text             0             0.0   
 EmailAddress     object             email             0             0.0   
 PhoneNumber      object             phone             0             0.0   
 StreetAddress    object              text             0             0.0   
 City             object              text             0             0.0   
 Country          object       categorical             0             0.0   
 ActiveFlag         bool           boolean             0             0.0   
 CreatedOn        object              date             0             0.0   
 CreatedByUser    object       categorical             0             0.0   
 ModifiedOn   

### 4.3 Text Profile

In [26]:
profile_results["text"]

{'SupplierName': {'WhitespaceCount': 1,
  'BlankCount': 0,
  'PotentialMissingMarkerCount': 0,
  'PotentialMissingMarkers': [],
  'MinimumLength': 15,
  'MaximumLength': 25,
  'AverageLength': np.float64(20.78)},
 'ContactPerson': {'WhitespaceCount': 0,
  'BlankCount': 0,
  'PotentialMissingMarkerCount': 0,
  'PotentialMissingMarkers': [],
  'MinimumLength': 6,
  'MaximumLength': 12,
  'AverageLength': np.float64(9.89)},
 'StreetAddress': {'WhitespaceCount': 0,
  'BlankCount': 0,
  'PotentialMissingMarkerCount': 0,
  'PotentialMissingMarkers': [],
  'MinimumLength': 5,
  'MaximumLength': 23,
  'AverageLength': np.float64(12.78)},
 'City': {'WhitespaceCount': 2,
  'BlankCount': 0,
  'PotentialMissingMarkerCount': 0,
  'PotentialMissingMarkers': [],
  'MinimumLength': 5,
  'MaximumLength': 9,
  'AverageLength': np.float64(5.89)}}

### 4.4 Numeric Profile

In [27]:
profile_results["numeric"]

{}

### 4.5 Date Profile

In [28]:
profile_results["date"]

{'CreatedOn': {'ValidDateCount': 9,
  'InvalidDateCount': 0,
  'AmbiguousDateCount': 0,
  'RecognizedFormats': ['ISO'],
  'MixedFormat': False,
  'MinimumDate': Timestamp('2026-07-01 00:00:00'),
  'MaximumDate': Timestamp('2026-07-05 00:00:00'),
  'FutureDateCount': 0},
 'ModifiedOn': {'ValidDateCount': 9,
  'InvalidDateCount': 0,
  'AmbiguousDateCount': 0,
  'RecognizedFormats': ['ISO'],
  'MixedFormat': False,
  'MinimumDate': Timestamp('2026-07-12 00:00:00'),
  'MaximumDate': Timestamp('2026-07-15 00:00:00'),
  'FutureDateCount': 0}}

### 4.6 Data-Quality Issues

In [14]:
profile_results["issues"]

,Column,IssueType,Severity,Count,Description
0,None,ExactDuplicateRows,High,1,Exact duplicate rows detected.
1,SupplierCode,DuplicateIdentifier,High,1,Repeated identifier values detected.
2,EmailAddress,DuplicateEmail,Medium,1,Duplicate email values detected.
3,City,Whitespace,Low,2,Leading or trailing whitespace detected.
4,SupplierName,Whitespace,Low,1,Leading or trailing whitespace detected.


## 5. Profiling Findings

The raw Supplier dataset contains **9 rows and 13 columns**.

The profiler detected the following semantic field types:

- `SupplierCode` → identifier
- `SupplierName`, `ContactPerson`, `StreetAddress`, `City` → text
- `EmailAddress` → email
- `PhoneNumber` → phone
- `Country`, `CreatedByUser`, `ModifiedByUser` → categorical
- `ActiveFlag` → boolean
- `CreatedOn`, `ModifiedOn` → date

### Data-quality findings

The raw dataset contains:

- **0 missing values**
- **1 exact duplicate row**
- **1 duplicate SupplierCode**
- **1 duplicate EmailAddress**
- **2 City whitespace issues**
- **1 SupplierName whitespace issue**

No potential missing-value markers were detected in the actual Supplier text fields.

The date fields were valid ISO-formatted dates with no invalid or ambiguous values.

These findings will be used to define the next stage: **data transformation and cleaning**.

## 6. Profiling Decision

The profiling stage is complete.

The profiler has identified the data-quality conditions that require treatment during transformation. The raw dataset itself remains unchanged.

### Transformation requirements identified

1. Remove the exact duplicate record according to the agreed deduplication rule.
2. Trim leading/trailing whitespace from affected text fields.
3. Investigate and resolve the duplicate `SupplierCode`.
4. Investigate and resolve the duplicate `EmailAddress`.
5. Preserve valid categorical repetition such as `Country = UAE`; repeated categorical values are not automatically errors.
6. Keep the raw dataset unchanged and create a separate cleaned/transformed dataset.

**Next stage: Data Transformation.**

## 7. Data Transformation & Cleaning

### 7.1 Create Transformation Dataset

The raw supplier dataset is preserved unchanged.

A separate working dataframe is created for all transformation and cleaning operations. This maintains a clear separation between the original source data and the transformed dataset.

In [15]:
# Create a separate working copy for transformation
procurement_supplier_master_clean = procurement_supplier_master_raw.copy()

print("Raw rows:", len(procurement_supplier_master_raw))
print("Transformation rows:", len(procurement_supplier_master_clean))

Raw rows: 9
Transformation rows: 9


### 7.2 Remove Exact Duplicate Rows

The profiler identified one exact duplicate row.

Because the duplicate is identical across all columns, it can be safely removed from the transformation dataset while preserving the original raw dataset for auditability.

In [16]:
# Identify exact duplicate rows before removing them
exact_duplicates = procurement_supplier_master_clean[
    procurement_supplier_master_clean.duplicated(keep=False)
]

print("Exact duplicate rows found:", len(exact_duplicates))
display(exact_duplicates)

Exact duplicate rows found: 2


,SupplierCode,SupplierName,ContactPerson,EmailAddress,PhoneNumber,StreetAddress,City,Country,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
6,SUP007,Nova Tech Distribution,Daniel Cruz,nova.tech@email.com,+971 50 111 2207,Sharjah Industrial Area,sharjah,UAE,True,2026-07-04,procurement_admin,2026-07-15,procurement_admin
8,SUP007,Nova Tech Distribution,Daniel Cruz,nova.tech@email.com,+971 50 111 2207,Sharjah Industrial Area,sharjah,UAE,True,2026-07-04,procurement_admin,2026-07-15,procurement_admin


In [17]:
# Remove exact duplicate rows
procurement_supplier_master_clean = procurement_supplier_master_clean.drop_duplicates()

# Check the new row count
print("Rows after removing exact duplicates:", len(procurement_supplier_master_clean))

Rows after removing exact duplicates: 8


In [18]:
remaining_duplicates = procurement_supplier_master_clean.duplicated().sum()

print("Remaining exact duplicate rows:", remaining_duplicates)

Remaining exact duplicate rows: 0


### 7.3 Standardize Text Fields

The profiling and inspection stages identified text consistency issues in the
Supplier dataset.

The following standardization rules are applied:

- Remove leading and trailing whitespace from `SupplierName`.
- Remove leading and trailing whitespace from `City`.
- Standardize `City` values to title case for consistent grouping, filtering,
  and reporting.

Examples:

- ` TechWorld Middle East ` → `TechWorld Middle East`
- ` sharjah ` → `Sharjah`

The transformations are limited to fields where a clear standardization rule
is appropriate. Other business text fields are preserved unless a specific
data-quality issue has been identified.

In [19]:
# Inspect SupplierName whitespace issues

supplier_name_whitespace = procurement_supplier_master_clean.loc[
    procurement_supplier_master_clean["SupplierName"].astype(str).str.strip()
    != procurement_supplier_master_clean["SupplierName"].astype(str),
    ["SupplierCode", "SupplierName"]
]

display(supplier_name_whitespace)

,SupplierCode,SupplierName
1,SUP002,TechWorld Middle East


In [20]:
# Inspect City whitespace issues

city_whitespace = procurement_supplier_master_clean.loc[
    procurement_supplier_master_clean["City"].astype(str).str.strip()
    != procurement_supplier_master_clean["City"].astype(str),
    ["SupplierCode", "City"]
]

display(city_whitespace)

,SupplierCode,City
6,SUP007,sharjah


#### Inspect Whitespace Characters

The affected values are displayed using `repr()` so that leading or trailing
whitespace is visible during inspection.

This allows the transformation to be verified against the actual source values
before any changes are applied.

In [21]:
# Display affected text values with whitespace made visible

print("SupplierName whitespace issues:")
for value in supplier_name_whitespace["SupplierName"]:
    print(repr(value))

print("\nCity whitespace issues:")
for value in city_whitespace["City"]:
    print(repr(value))

SupplierName whitespace issues:
' TechWorld Middle East '

City whitespace issues:
' sharjah '


#### Text Standardization Rule

The inspection confirmed that the affected `SupplierName` and `City` values
contain leading and trailing whitespace.

The transformation will use `str.strip()` to remove whitespace from the
beginning and end of the affected text values.

This will not alter meaningful characters or internal spaces within the
business values.

Only `procurement_supplier_master_clean` will be modified. The original `procurement_supplier_master_raw` dataset
will remain unchanged.

In [22]:
# Apply text standardization

procurement_supplier_master_clean["SupplierName"] = (
    procurement_supplier_master_clean["SupplierName"].str.strip()
)

procurement_supplier_master_clean["City"] = (
    procurement_supplier_master_clean["City"]
    .str.strip()
    .str.title()
)

#### Validate Text Standardization

The transformed `SupplierName` and `City` fields are checked again for
leading or trailing whitespace.

The expected result is zero remaining whitespace issues in both fields.

In [23]:
# Validate whitespace cleanup

supplier_name_whitespace_after = (
    procurement_supplier_master_clean["SupplierName"].astype(str).str.strip()
    != procurement_supplier_master_clean["SupplierName"].astype(str)
).sum()

city_whitespace_after = (
    procurement_supplier_master_clean["City"].astype(str).str.strip()
    != procurement_supplier_master_clean["City"].astype(str)
).sum()

print("SupplierName whitespace issues after:", supplier_name_whitespace_after)
print("City whitespace issues after:", city_whitespace_after)

SupplierName whitespace issues after: 0
City whitespace issues after: 0


### 7.4 Investigate Duplicate SupplierCode

The profiling stage identified one duplicate `SupplierCode`.

Unlike an exact duplicate row, a repeated SupplierCode cannot be removed
automatically because it may represent:

- the same supplier recorded more than once,
- a legitimate business relationship,
- a conflicting supplier record,
- or another source-data issue.

The duplicate SupplierCode records will therefore be inspected before any
transformation decision is made.

No record will be deleted at this stage.

In [24]:
# Inspect duplicate SupplierCode records

duplicate_supplier_codes = procurement_supplier_master_clean[
    procurement_supplier_master_clean["SupplierCode"].duplicated(keep=False)
].sort_values("SupplierCode")

display(duplicate_supplier_codes)

,SupplierCode,SupplierName,ContactPerson,EmailAddress,PhoneNumber,StreetAddress,City,Country,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser


In [25]:
# Inspect duplicate SupplierCode records

duplicate_supplier_codes = procurement_supplier_master_clean[
    procurement_supplier_master_clean["SupplierCode"].duplicated(keep=False)
].sort_values("SupplierCode")

display(
    duplicate_supplier_codes[
        [
            "SupplierCode",
            "SupplierName",
            "ContactPerson",
            "EmailAddress",
            "PhoneNumber",
            "StreetAddress",
            "City",
            "Country",
            "ActiveFlag",
            "CreatedOn",
            "CreatedByUser",
            "ModifiedOn",
            "ModifiedByUser"
        ]
    ]
)

,SupplierCode,SupplierName,ContactPerson,EmailAddress,PhoneNumber,StreetAddress,City,Country,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser


#### Duplicate SupplierCode Finding

The profiling stage initially identified one duplicate `SupplierCode`.

During Section 7.2, the duplicate `SupplierCode` was found to belong to
two completely identical supplier records. The duplicate record was
removed as an exact duplicate.

After the exact duplicate was removed, the `SupplierCode` duplication was
also resolved.

Therefore, no additional transformation is required for `SupplierCode`.

#### Validate SupplierCode Uniqueness

The duplicate `SupplierCode` identified during profiling was caused by the
exact duplicate record removed in Section 7.2.

The transformed dataset is checked again to confirm that no duplicate
`SupplierCode` values remain.

Expected result:

- Duplicate `SupplierCode` count → 0

In [26]:
# Validate SupplierCode uniqueness

duplicate_supplier_code_count = (
    procurement_supplier_master_clean["SupplierCode"].duplicated().sum()
)

print(
    "Duplicate SupplierCode count after transformation:",
    duplicate_supplier_code_count
)

Duplicate SupplierCode count after transformation: 0


### 7.5 Investigate Duplicate EmailAddress

The profiling stage identified one duplicate `EmailAddress`.

As with `SupplierCode`, the duplicate email value may have been caused by
the exact duplicate record removed in Section 7.2, or it may represent a
legitimate shared contact or another data-quality issue.

The transformed dataset will therefore be checked to determine whether any
duplicate email addresses remain.

No records will be deleted based on email duplication alone.

In [27]:
# Check for remaining duplicate EmailAddress values

duplicate_email_count = (
    procurement_supplier_master_clean["EmailAddress"].duplicated().sum()
)

print(
    "Duplicate EmailAddress count after transformation:",
    duplicate_email_count
)

Duplicate EmailAddress count after transformation: 0


### 7.6 Standardize Date Fields

The profiling stage confirmed that `CreatedOn` and `ModifiedOn` contain valid,
unambiguous ISO-formatted dates with no invalid values.

The transformation will therefore focus on converting these fields to proper
datetime data types for consistent downstream processing, analysis, and
database loading.

No date values will be altered because the existing date values are already
valid.

The original `procurement_supplier_master_raw` dataset will remain unchanged.

In [28]:
# Inspect current date field data types

print("CreatedOn dtype:", procurement_supplier_master_clean["CreatedOn"].dtype)
print("ModifiedOn dtype:", procurement_supplier_master_clean["ModifiedOn"].dtype)

CreatedOn dtype: object
ModifiedOn dtype: object


In [29]:
# Convert date fields to datetime

procurement_supplier_master_clean["CreatedOn"] = pd.to_datetime(
    procurement_supplier_master_clean["CreatedOn"]
)

procurement_supplier_master_clean["ModifiedOn"] = pd.to_datetime(
    procurement_supplier_master_clean["ModifiedOn"]
)

In [30]:
# Validate date field data types

print("CreatedOn dtype:", procurement_supplier_master_clean["CreatedOn"].dtype)
print("ModifiedOn dtype:", procurement_supplier_master_clean["ModifiedOn"].dtype)

CreatedOn dtype: datetime64[ns]
ModifiedOn dtype: datetime64[ns]


#### Validate Date Fields

The date fields have been converted to proper datetime types.

The transformed values are now validated to confirm that:

- No invalid dates were introduced.
- No missing datetime values were introduced.
- The date fields remain suitable for downstream database loading and analysis.

In [31]:
# Validate date values

created_on_missing = procurement_supplier_master_clean["CreatedOn"].isna().sum()
modified_on_missing = procurement_supplier_master_clean["ModifiedOn"].isna().sum()

print("CreatedOn missing datetime values:", created_on_missing)
print("ModifiedOn missing datetime values:", modified_on_missing)

CreatedOn missing datetime values: 0
ModifiedOn missing datetime values: 0


## 8. Transformation Validation

The transformation stage is validated against the data-quality findings
identified during profiling.

The purpose of this stage is to confirm that the identified issues were
addressed without introducing new data-quality problems.

The validation compares the known profiling findings before transformation
with the resulting state of `procurement_supplier_master_clean`.

Key validation areas include:

- Exact duplicate rows
- `SupplierName` whitespace
- `City` whitespace
- Duplicate `SupplierCode`
- Duplicate `EmailAddress`
- Missing values
- Date validity and data types
- Final row and column counts

The original `procurement_supplier_master_raw` dataset remains preserved for comparison and
auditability.

### 8.1 Validate Dataset Structure

The transformed dataset is checked against the original raw dataset to confirm
the expected structural change.

The transformation should:

- Reduce the row count from 9 to 8 because one exact duplicate was removed.
- Preserve all 13 original columns.
- Keep the original `procurement_supplier_master_raw` dataset unchanged.

In [32]:
# Validate dataset structure

print("Raw rows:", len(procurement_supplier_master_raw))
print("Clean rows:", len(procurement_supplier_master_clean))

print("Raw columns:", len(procurement_supplier_master_raw.columns))
print("Clean columns:", len(procurement_supplier_master_clean.columns))

Raw rows: 9
Clean rows: 8
Raw columns: 13
Clean columns: 13


### 8.2 Validate Exact Duplicate Removal

The transformed dataset is checked to confirm that the exact duplicate
identified during profiling has been completely removed.

The expected result is zero remaining exact duplicate rows.

In [33]:
# Validate exact duplicate removal

exact_duplicate_count = procurement_supplier_master_clean.duplicated().sum()

print(
    "Remaining exact duplicate rows:",
    exact_duplicate_count
)

Remaining exact duplicate rows: 0


### 8.3 Validate Text Standardization

The transformed `SupplierName` and `City` fields are checked to confirm that
the leading and trailing whitespace identified during profiling has been
removed.

Expected result:

- `SupplierName` whitespace issues → 0
- `City` whitespace issues → 0

In [34]:
# Validate text standardization

supplier_name_whitespace_count = (
    procurement_supplier_master_clean["SupplierName"].astype(str).str.strip()
    != procurement_supplier_master_clean["SupplierName"].astype(str)
).sum()

city_whitespace_count = (
    procurement_supplier_master_clean["City"].astype(str).str.strip()
    != procurement_supplier_master_clean["City"].astype(str)
).sum()

city_capitalization_count = (
    procurement_supplier_master_clean["City"].astype(str)
    != procurement_supplier_master_clean["City"].astype(str).str.title()
).sum()

print(
    "SupplierName whitespace issues:",
    supplier_name_whitespace_count
)

print(
    "City whitespace issues:",
    city_whitespace_count
)

print(
    "City capitalization issues:",
    city_capitalization_count
)

SupplierName whitespace issues: 0
City whitespace issues: 0
City capitalization issues: 0


### 8.4 Validate SupplierCode Integrity

The profiling stage initially identified one duplicate `SupplierCode`.
Investigation confirmed that the duplicate was caused by the exact duplicate
record removed in Section 7.2.

The transformed dataset is therefore checked to confirm that all remaining
`SupplierCode` values are unique.

Expected result:

- Duplicate `SupplierCode` count → 0

In [35]:
# Validate SupplierCode uniqueness

duplicate_supplier_code_count = (
    procurement_supplier_master_clean["SupplierCode"].duplicated().sum()
)

print(
    "Duplicate SupplierCode count:",
    duplicate_supplier_code_count
)

Duplicate SupplierCode count: 0


### 8.5 Validate EmailAddress Integrity

The profiling stage initially identified one duplicate `EmailAddress`.

Investigation confirmed that the duplicate email belonged to the exact
duplicate record removed in Section 7.2.

The transformed dataset is therefore checked to confirm that no duplicate
`EmailAddress` values remain.

Expected result:

- Duplicate `EmailAddress` count → 0

In [36]:
# Validate EmailAddress uniqueness

duplicate_email_count = (
    procurement_supplier_master_clean["EmailAddress"].duplicated().sum()
)

print(
    "Duplicate EmailAddress count:",
    duplicate_email_count
)

Duplicate EmailAddress count: 0


### 8.6 Validate Missing Values

The profiling stage identified zero missing values across the Supplier
dataset.

The transformed dataset is checked again to confirm that the transformation
process did not introduce any new missing values.

Expected result:

- Total missing values → 0

In [37]:
# Validate missing values

total_missing_values = procurement_supplier_master_clean.isna().sum().sum()

print(
    "Total missing values:",
    total_missing_values
)

Total missing values: 0


### 8.7 Validate Date Integrity

The `CreatedOn` and `ModifiedOn` fields were converted to proper datetime
types during transformation.

The transformed dataset is checked to confirm:

- Both fields remain proper datetime types.
- No missing datetime values are present.
- No invalid datetime values were introduced.

The original profiling results showed that both fields contained valid,
unambiguous ISO-formatted dates.

In [38]:
# Validate date integrity

created_on_invalid = procurement_supplier_master_clean["CreatedOn"].isna().sum()
modified_on_invalid = procurement_supplier_master_clean["ModifiedOn"].isna().sum()

print("CreatedOn dtype:", procurement_supplier_master_clean["CreatedOn"].dtype)
print("ModifiedOn dtype:", procurement_supplier_master_clean["ModifiedOn"].dtype)

print("CreatedOn invalid/missing dates:", created_on_invalid)
print("ModifiedOn invalid/missing dates:", modified_on_invalid)

CreatedOn dtype: datetime64[ns]
ModifiedOn dtype: datetime64[ns]
CreatedOn invalid/missing dates: 0
ModifiedOn invalid/missing dates: 0


### 8.8 Final Transformation Validation Summary

The transformation and validation results are summarized against the
data-quality findings identified during the profiling stage.

The results demonstrate that the identified issues were addressed without
introducing new missing values, invalid dates, duplicate identifiers, or
structural inconsistencies.

The original `procurement_supplier_master_raw` dataset remains preserved, while `procurement_supplier_master_clean`
represents the validated transformed dataset.

In [39]:
# Build final transformation validation summary

validation_summary = pd.DataFrame([
    {
        "Validation": "Row count",
        "Before": len(procurement_supplier_master_raw),
        "After": len(procurement_supplier_master_clean),
        "Status": "Passed" if len(procurement_supplier_master_clean) == 8 else "Review"
    },
    {
        "Validation": "Column count",
        "Before": len(procurement_supplier_master_raw.columns),
        "After": len(procurement_supplier_master_clean.columns),
        "Status": "Passed"
        if len(procurement_supplier_master_raw.columns) == len(procurement_supplier_master_clean.columns)
        else "Review"
    },
    {
        "Validation": "Exact duplicate rows",
        "Before": 1,
        "After": procurement_supplier_master_clean.duplicated().sum(),
        "Status": "Passed"
        if procurement_supplier_master_clean.duplicated().sum() == 0
        else "Review"
    },
    {
        "Validation": "SupplierName whitespace",
        "Before": 1,
        "After": (
            procurement_supplier_master_clean["SupplierName"].astype(str).str.strip()
            != procurement_supplier_master_clean["SupplierName"].astype(str)
        ).sum(),
        "Status": "Passed"
        if (
            procurement_supplier_master_clean["SupplierName"].astype(str).str.strip()
            != procurement_supplier_master_clean["SupplierName"].astype(str)
        ).sum() == 0
        else "Review"
    },
    {
        "Validation": "City whitespace",
        "Before": 2,
        "After": (
            procurement_supplier_master_clean["City"].astype(str).str.strip()
            != procurement_supplier_master_clean["City"].astype(str)
        ).sum(),
        "Status": "Passed"
        if (
            procurement_supplier_master_clean["City"].astype(str).str.strip()
            != procurement_supplier_master_clean["City"].astype(str)
        ).sum() == 0
        else "Review"
    },
    {
        "Validation": "City capitalization",
        "Before": 1,
        "After": (
            procurement_supplier_master_clean["City"].astype(str)
            != procurement_supplier_master_clean["City"].astype(str).str.title()
        ).sum(),
        "Status": "Passed"
        if (
            procurement_supplier_master_clean["City"].astype(str)
            != procurement_supplier_master_clean["City"].astype(str).str.title()
        ).sum() == 0
        else "Review"
    },
    {
        "Validation": "Duplicate SupplierCode",
        "Before": 1,
        "After": procurement_supplier_master_clean["SupplierCode"].duplicated().sum(),
        "Status": "Passed"
        if procurement_supplier_master_clean["SupplierCode"].duplicated().sum() == 0
        else "Review"
    },
    {
        "Validation": "Duplicate EmailAddress",
        "Before": 1,
        "After": procurement_supplier_master_clean["EmailAddress"].duplicated().sum(),
        "Status": "Passed"
        if procurement_supplier_master_clean["EmailAddress"].duplicated().sum() == 0
        else "Review"
    },
    {
        "Validation": "Missing values",
        "Before": procurement_supplier_master_raw.isna().sum().sum(),
        "After": procurement_supplier_master_clean.isna().sum().sum(),
        "Status": "Passed"
        if procurement_supplier_master_clean.isna().sum().sum() == 0
        else "Review"
    },
    {
        "Validation": "CreatedOn invalid/missing",
        "Before": 0,
        "After": procurement_supplier_master_clean["CreatedOn"].isna().sum(),
        "Status": "Passed"
        if procurement_supplier_master_clean["CreatedOn"].isna().sum() == 0
        else "Review"
    },
    {
        "Validation": "ModifiedOn invalid/missing",
        "Before": 0,
        "After": procurement_supplier_master_clean["ModifiedOn"].isna().sum(),
        "Status": "Passed"
        if procurement_supplier_master_clean["ModifiedOn"].isna().sum() == 0
        else "Review"
    }
])

display(validation_summary)

,Validation,Before,After,Status
0,Row count,9,8,Passed
1,Column count,13,13,Passed
2,Exact duplicate rows,1,0,Passed
3,SupplierName whitespace,1,0,Passed
4,City whitespace,2,0,Passed
5,City capitalization,1,0,Passed
6,Duplicate SupplierCode,1,0,Passed
7,Duplicate EmailAddress,1,0,Passed
8,Missing values,0,0,Passed
9,CreatedOn invalid/missing,0,0,Passed


## 9. Clean Supplier Dataset

The `procurement_supplier_master_clean` dataframe represents the final transformed and validated
Supplier dataset.

At this stage, the dataset has passed the transformation validation checks
and is ready for downstream use.

Final characteristics:

- 8 supplier records
- 13 columns
- No exact duplicate rows
- No duplicate `SupplierCode` values
- No duplicate `EmailAddress` values
- No remaining identified whitespace issues
- No missing values
- `CreatedOn` and `ModifiedOn` represented as datetime fields

The original `procurement_supplier_master_raw` dataframe remains preserved as the raw source
dataset.

The validated `procurement_supplier_master_clean` dataframe will be used for downstream export,
PostgreSQL loading, SQL/data modeling, and Power BI development.

### 9.1 Final Clean Dataset Inspection

The final `procurement_supplier_master_clean` dataframe is displayed for a last visual inspection
before export.

This confirms that the transformed dataset contains the expected records,
columns, standardized text values, and datetime fields after all cleaning and
validation steps.

In [40]:
# Display the final validated Supplier dataset

display(procurement_supplier_master_clean)

,SupplierCode,SupplierName,ContactPerson,EmailAddress,PhoneNumber,StreetAddress,City,Country,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,SUP001,ABC Trading LLC,John Smith,supplier1@email.com,+971 50 111 2201,Al Quoz,Dubai,UAE,True,2026-07-01,procurement_admin,2026-07-15,procurement_admin
1,SUP002,TechWorld Middle East,Maria Khan,sales@techworld.ae,+971501112202,Jebel Ali,Dubai,UAE,True,2026-07-01,procurement_admin,2026-07-14,procurement_admin
2,SUP003,Gulf Electronics FZCO,Ahmed Rahman,CONTACT@GULFELECTRONICS.AE,+971 50 111 2203,Dubai Silicon Oasis,Dubai,UAE,True,2026-07-02,procurement_admin,2026-07-15,procurement_admin
3,SUP004,Asia Components Trading,Li Wei,asia.components@email.com,+971 50 111 2204,Deira,Dubai,UAE,True,2026-07-02,procurement_admin,2026-07-13,procurement_admin
4,SUP005,Global Devices Ltd,Sarah Lee,global.devices@email.com,+971 50 111 2205,Business Bay,Dubai,UAE,True,2026-07-03,procurement_admin,2026-07-15,procurement_admin
5,SUP006,Emirates IT Supplies,Omar Ali,sales@emiratesit.ae,+971 50 111 2206,Mussafah,Abu Dhabi,UAE,True,2026-07-03,procurement_admin,2026-07-15,procurement_admin
6,SUP007,Nova Tech Distribution,Daniel Cruz,nova.tech@email.com,+971 50 111 2207,Sharjah Industrial Area,Sharjah,UAE,True,2026-07-04,procurement_admin,2026-07-15,procurement_admin
7,SUP008,Prime Accessories Trading,Nadia Hassan,prime.acc@email.com,+971 50 111 2208,Al Qusais,Dubai,UAE,False,2026-07-05,procurement_admin,2026-07-12,procurement_admin


## 10. Export / Load

The validated `procurement_supplier_master_clean` dataframe is now ready for downstream
consumption.

The cleaned dataset will be exported as a CSV file while preserving the
validated structure and transformations completed in Sections 7–9.

This exported dataset will serve as the handoff from the Python ETL stage to
the downstream PostgreSQL data-loading and SQL/data-modeling stages.

The original `procurement_supplier_master_raw` dataset remains unchanged and is not overwritten.

### 10.1 Export Clean Supplier Dataset

The validated `procurement_supplier_master_clean` dataframe is exported as a CSV file for
downstream database loading.

The exported file represents the final validated Supplier dataset and will
serve as the handoff from the Python ETL stage to PostgreSQL.

The raw source dataset is preserved separately and is not overwritten.

In [41]:
# =========================================================
# 10.1 Export Clean Supplier Dataset
# =========================================================

from pathlib import Path

clean_supplier_path = Path(
    r"C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce"
    r"\data\02_Procurement_ERP\clean\procurement_supplier_master_clean.csv"
)

# Create the clean directory if it does not exist
clean_supplier_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

procurement_supplier_master_clean.to_csv(
    clean_supplier_path,
    index=False
)

print("Supplier clean dataset exported successfully.")
print("Path:", clean_supplier_path)
print("Rows exported:", len(procurement_supplier_master_clean))
print("Columns exported:", len(procurement_supplier_master_clean.columns))


Supplier clean dataset exported successfully.
Path: C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\02_Procurement_ERP\clean\procurement_supplier_master_clean.csv
Rows exported: 8
Columns exported: 13


### 10.2 Validate Exported CSV

The exported CSV file is read back into a new dataframe to verify that the
file contains the expected number of rows and columns.

This provides a final check that the dataset written to disk matches the
validated `procurement_supplier_master_clean` dataframe before database loading.

Expected result:

- 8 rows
- 13 columns

In [42]:
# Read back the exported CSV for validation

supplier_exported = pd.read_csv(
    clean_supplier_path
)

print("Exported rows:", len(supplier_exported))
print("Exported columns:", len(supplier_exported.columns))


Exported rows: 8
Exported columns: 13


In [43]:
# Validate exported CSV column structure

columns_match = supplier_exported.columns.equals(
    procurement_supplier_master_clean.columns
)

print("Column structure matches:", columns_match)

Column structure matches: True


In [44]:
# Show the exact location of the exported CSV

print("Supplier CSV location:")
print(clean_supplier_path.resolve())


Supplier CSV location:
C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\02_Procurement_ERP\clean\procurement_supplier_master_clean.csv


### 10.3 Prepare PostgreSQL Load Dataset

The validated `procurement_supplier_master_clean` dataframe is mapped to the established
PostgreSQL target schema.

The source-style column names are converted to the target column names
defined for `commerce.supplier`.

A separate `procurement_supplier_master_db_ready` dataframe is created so that the validated
`procurement_supplier_master_clean` dataset remains unchanged.

This database-ready dataset will be used for the PostgreSQL loading stage.

In [70]:
# Map validated Supplier fields to the PostgreSQL target schema

procurement_supplier_master_db_ready = procurement_supplier_master_clean.rename(columns={
    "SupplierCode": "supplier_id",
    "SupplierName": "supplier_name",
    "ContactPerson": "contact_person",
    "EmailAddress": "email",
    "PhoneNumber": "phone",
    "StreetAddress": "address",
    "City": "city",
    "Country": "country",
    "ActiveFlag": "is_active",
    "CreatedOn": "created_date",
    "CreatedByUser": "created_by",
    "ModifiedOn": "updated_date",
    "ModifiedByUser": "updated_by"
}).copy()

print("Database-ready rows:", len(procurement_supplier_master_db_ready))
print("Database-ready columns:", len(procurement_supplier_master_db_ready.columns))

Database-ready rows: 8
Database-ready columns: 13


### 10.4 Validate PostgreSQL Load Dataset

The database-ready dataframe is validated against the established
`commerce.supplier` target schema.

This confirms that the source-to-target column mapping produced the expected
13 PostgreSQL column names in the correct order before the data is loaded.

In [71]:
# Validate database-ready column structure

expected_supplier_columns = [
    "supplier_id",
    "supplier_name",
    "contact_person",
    "email",
    "phone",
    "address",
    "city",
    "country",
    "is_active",
    "created_date",
    "created_by",
    "updated_date",
    "updated_by"
]

columns_match = list(procurement_supplier_master_db_ready.columns) == expected_supplier_columns

print("Column mapping matches PostgreSQL target:", columns_match)
print("Rows ready for PostgreSQL:", len(procurement_supplier_master_db_ready))
print("Columns ready for PostgreSQL:", len(procurement_supplier_master_db_ready.columns))

Column mapping matches PostgreSQL target: True
Rows ready for PostgreSQL: 8
Columns ready for PostgreSQL: 13


## 11. PostgreSQL Load

The validated Supplier dataset is now ready for loading into the centralized
PostgreSQL database.

The PostgreSQL stage will map the source-style Supplier fields produced by the
Python ETL process to the established target database schema.

The existing `customer` table is already completed and will not be recreated
or reloaded.

Before creating or loading the Supplier target table, the existing PostgreSQL
schema and database design will be inspected to ensure that the implementation
matches the established LJ Dev Commerce architecture.

### 11.1 Inspect Existing PostgreSQL Schema

Before loading the Supplier dataset, the existing PostgreSQL database structure
is inspected to confirm the current schema, tables, columns, data types, and
constraints.

This prevents the ETL process from creating duplicate tables or introducing a
target structure that conflicts with the established LJ Dev Commerce database
architecture.

The existing `customer` table is expected to be present and will be preserved.

### Database Preparation and Target Definition

The PostgreSQL database preparation and Supplier target-table definition for
Sections 11.1–11.5 are documented in the accompanying `supplier_load.sql`
script.

The SQL script covers:

- **11.1 — Inspect Existing PostgreSQL Schema:** Confirm the existing tables
  within the `commerce` schema.
- **11.2 — Inspect Existing Customer Table:** Review the existing Customer
  table structure and PostgreSQL data-type conventions.
- **11.2b — Inspect Existing Customer Constraints:** Identify the Customer
  table's primary key and other constraints.
- **11.3 — Supplier Source-to-Target Mapping:** Document the mapping between
  the validated Python Supplier fields and the PostgreSQL `commerce.supplier`
  target schema.
- **11.4 — Create Supplier Target Table:** Create the `commerce.supplier`
  table according to the established database architecture.
- **11.4b — Verify Supplier Target Table:** Confirm the Supplier table
  columns, data types, and nullability.
- **11.5 — Verify Supplier Primary Key:** Confirm that `supplier_id` is
  defined as the primary key.

These steps establish and verify the PostgreSQL target before any Supplier
records are loaded.

The actual data loading is performed from Python using the validated
`procurement_supplier_master_db_ready` dataset.

### 11.6 PostgreSQL Connection

Establish a connection from the Python notebook to the PostgreSQL database.

The connection will be used to load the validated `procurement_supplier_master_db_ready`
dataset into the existing `commerce.supplier` target table.

No Supplier records are inserted at this stage. This step only establishes
and verifies the database connection.

In [76]:
# Establish PostgreSQL connection

import psycopg2
from getpass import getpass

password = getpass("PostgreSQL password: ")

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="lj_dev_commerce",
    user="postgres",
    password=password
)

cursor = conn.cursor()

print("PostgreSQL connection established successfully.")

PostgreSQL connection established successfully.


In [77]:
# Verify PostgreSQL-ready Supplier dataset

print("Rows:", len(procurement_supplier_master_db_ready))
print("Columns:", len(procurement_supplier_master_db_ready.columns))
print("Columns:")
print(procurement_supplier_master_db_ready.columns.tolist())

Rows: 8
Columns: 13
Columns:
['supplier_id', 'supplier_name', 'contact_person', 'email', 'phone', 'address', 'city', 'country', 'is_active', 'created_date', 'created_by', 'updated_date', 'updated_by']


### 11.7 Prepare Parameterized INSERT

The validated `procurement_supplier_master_db_ready` DataFrame is already aligned with the
`commerce.supplier` PostgreSQL target schema.

A parameterized INSERT statement is prepared so that Python can pass the
Supplier values separately from the SQL command.

No records are inserted in this step. The actual database load will be
performed in the following step.

In [78]:
# Prepare parameterized INSERT statement

insert_sql = """
INSERT INTO commerce.supplier (
    supplier_id,
    supplier_name,
    contact_person,
    email,
    phone,
    address,
    city,
    country,
    is_active,
    created_date,
    created_by,
    updated_date,
    updated_by
)
VALUES (
    %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s
);
"""

print("Parameterized INSERT statement prepared successfully.")

Parameterized INSERT statement prepared successfully.


### 11.8 Load Supplier Data

Execute the parameterized INSERT statement for each record in the validated
`procurement_supplier_master_db_ready` DataFrame.

The 8 validated Supplier records are loaded into the existing
`commerce.supplier` PostgreSQL table.

The transaction is committed only after the insert operations complete
successfully.

In [79]:
# Load validated Supplier data into PostgreSQL

supplier_rows = list(
    procurement_supplier_master_db_ready.itertuples(index=False, name=None)
)

cursor.executemany(
    insert_sql,
    supplier_rows
)

conn.commit()

print(
    "Supplier data loaded successfully.",
    "Rows inserted:",
    len(supplier_rows)
)

Supplier data loaded successfully. Rows inserted: 8


### 11.9 Verify PostgreSQL Row Count

Verify the number of records currently stored in the PostgreSQL
`commerce.supplier` target table.

The database row count is compared with the number of records loaded from
the validated `procurement_supplier_master_db_ready` DataFrame.

In [80]:
# Verify Supplier row count in PostgreSQL

cursor.execute("""
    SELECT COUNT(*)
    FROM commerce.supplier;
""")

supplier_db_count = cursor.fetchone()[0]

print("Supplier rows in PostgreSQL:", supplier_db_count)
print("Supplier rows expected:", len(procurement_supplier_master_db_ready))

print(
    "Row count validation:",
    "Passed" if supplier_db_count == len(procurement_supplier_master_db_ready) else "Review"
)

Supplier rows in PostgreSQL: 8
Supplier rows expected: 8
Row count validation: Passed


### 11.10 Verify Loaded Supplier Data

Retrieve the Supplier records from the PostgreSQL target table and compare
the loaded data with the validated `procurement_supplier_master_db_ready` dataset.

This confirms that the Supplier records were loaded into the correct target
columns and that the values were preserved during the Python-to-PostgreSQL
load process.

In [81]:
# Retrieve loaded Supplier data from PostgreSQL

cursor.execute("""
    SELECT
        supplier_id,
        supplier_name,
        contact_person,
        email,
        phone,
        address,
        city,
        country,
        is_active,
        created_date,
        created_by,
        updated_date,
        updated_by
    FROM commerce.supplier
    ORDER BY supplier_id;
""")

supplier_loaded = cursor.fetchall()

supplier_loaded_df = pd.DataFrame(
    supplier_loaded,
    columns=procurement_supplier_master_db_ready.columns
)

print("Loaded Supplier rows:", len(supplier_loaded_df))

display(supplier_loaded_df)

Loaded Supplier rows: 8


,supplier_id,supplier_name,contact_person,email,phone,address,city,country,is_active,created_date,created_by,updated_date,updated_by
0,SUP001,ABC Trading LLC,John Smith,supplier1@email.com,+971 50 111 2201,Al Quoz,Dubai,UAE,True,2026-07-01,procurement_admin,2026-07-15,procurement_admin
1,SUP002,TechWorld Middle East,Maria Khan,sales@techworld.ae,+971501112202,Jebel Ali,Dubai,UAE,True,2026-07-01,procurement_admin,2026-07-14,procurement_admin
2,SUP003,Gulf Electronics FZCO,Ahmed Rahman,CONTACT@GULFELECTRONICS.AE,+971 50 111 2203,Dubai Silicon Oasis,Dubai,UAE,True,2026-07-02,procurement_admin,2026-07-15,procurement_admin
3,SUP004,Asia Components Trading,Li Wei,asia.components@email.com,+971 50 111 2204,Deira,Dubai,UAE,True,2026-07-02,procurement_admin,2026-07-13,procurement_admin
4,SUP005,Global Devices Ltd,Sarah Lee,global.devices@email.com,+971 50 111 2205,Business Bay,Dubai,UAE,True,2026-07-03,procurement_admin,2026-07-15,procurement_admin
5,SUP006,Emirates IT Supplies,Omar Ali,sales@emiratesit.ae,+971 50 111 2206,Mussafah,Abu Dhabi,UAE,True,2026-07-03,procurement_admin,2026-07-15,procurement_admin
6,SUP007,Nova Tech Distribution,Daniel Cruz,nova.tech@email.com,+971 50 111 2207,Sharjah Industrial Area,Sharjah,UAE,True,2026-07-04,procurement_admin,2026-07-15,procurement_admin
7,SUP008,Prime Accessories Trading,Nadia Hassan,prime.acc@email.com,+971 50 111 2208,Al Qusais,Dubai,UAE,False,2026-07-05,procurement_admin,2026-07-12,procurement_admin


### 11.11 Final Source-to-Database Validation

Compare the validated `procurement_supplier_master_db_ready` DataFrame with the records retrieved
from the PostgreSQL `commerce.supplier` table.

This final validation confirms that the records loaded into PostgreSQL match
the database-ready dataset produced by the Python ETL process.

In [82]:
# Final source-to-database validation

source_df = (
    procurement_supplier_master_db_ready
    .sort_values("supplier_id")
    .reset_index(drop=True)
)

database_df = (
    supplier_loaded_df
    .sort_values("supplier_id")
    .reset_index(drop=True)
)

data_match = source_df.equals(database_df)

print("Source rows:", len(source_df))
print("Database rows:", len(database_df))
print("Source-to-database match:", data_match)

Source rows: 8
Database rows: 8
Source-to-database match: True


### 11.12 PostgreSQL Load Completion

The validated Supplier dataset has been successfully loaded into the
`commerce.supplier` PostgreSQL target table.

The load was performed using the validated `procurement_supplier_master_db_ready` DataFrame
through a parameterized Python-to-PostgreSQL INSERT process.

Final validation confirmed:

- 8 Supplier records loaded
- PostgreSQL row count matches the expected source count
- Loaded Supplier records were successfully retrieved
- Source-to-database comparison returned `True`

The Supplier dataset is now available in PostgreSQL for the subsequent
SQL analysis and data-modeling stages.